# 00 — Environment and data provenance

Pins down *what exactly* the rest of this series runs against: the interpreter and library versions, this checkout's git commit, the strategy/feature schema versions and config identity, and the SHA-256 identity of every raw CSV the pipeline reads.

**Data mode: REAL DATA.** Every figure below is computed from the genuine Bloomberg CSV exports for this strategy, through production AtlasQuant code. No synthetic, simulated, imputed, or approximated value appears anywhere in this notebook.

Those exports are gitignored, so they are absent in a fresh clone. When they are missing this notebook still executes top to bottom, but every computing cell prints an explicit *real data not present — cannot produce a genuine result* notice instead of a number. It never falls back to synthetic data: a plausible-looking figure derived from invented inputs is not a result (see `CLAUDE.md`).

In [ ]:
import _real_data as rd

print(rd.banner())
AVAILABLE = rd.data_available()


## Interpreter and library versions

In [ ]:
import platform
import sys

print("python           ", sys.version.split()[0], f"({platform.platform()})")

for module_name in ("lightgbm", "sklearn", "numpy", "pandas", "nbformat"):
    try:
        module = __import__(module_name)
        print(f"{module_name:<17}", getattr(module, "__version__", "?"))
    except ImportError:
        print(f"{module_name:<17}", "NOT INSTALLED")


## This checkout

In [ ]:
print("repo root  ", rd.REPO_ROOT)
print("git commit ", rd.git_commit() or "unavailable (not a git checkout, or git missing)")


## Strategy identity

The config identity is a deterministic hash of every resolved configuration value. Any change that could alter a result changes it.

In [ ]:
from atlas_quant.strategies.multi_factor_ranking_ml.config import (
    FEATURE_SCHEMA_VERSION,
    STRATEGY_ID,
    STRATEGY_VERSION,
)
from atlas_quant.strategies.multi_factor_ranking_ml.feature_domain import FEATURE_NAMES

config = rd.config()
print("strategy_id            ", STRATEGY_ID)
print("strategy_version       ", STRATEGY_VERSION)
print("feature_schema_version ", FEATURE_SCHEMA_VERSION)
print("feature count          ", len(FEATURE_NAMES))
print("config identity        ", config.identity())
print("model identity         ", config.model.identity())
print()
print("ml_train_years     ", config.ml_train_years)
print("min_train_quarters ", config.min_train_quarters)
print("n_relevance_grades ", config.n_relevance_grades)
print("return_cap         ", config.return_cap)
print("exclude_sectors    ", config.exclude_sectors)
print("cross_sectional_rank_features")
for name in config.cross_sectional_rank_features:
    print("                    ", name)
print("model              ", config.model)

## Raw source identity

SHA-256 of every CSV the pipeline reads, so any later drift in the underlying data is detectable rather than assumed away.

In [ ]:
if AVAILABLE:
    print(f"{'file':<32} {'size (MB)':>10}  {'modified':<20} sha256")
    for entry in rd.source_manifest():
        if not entry.present:
            print(f"{entry.name:<32} {'ABSENT':>10}")
            continue
        print(
            f"{entry.name:<32} {entry.size_bytes / 1e6:>10.1f}  "
            f"{entry.modified:%Y-%m-%d %H:%M:%S}  {entry.sha256[:32]}..."
        )
else:
    print(rd.DATA_UNAVAILABLE_MESSAGE)


## The bounded slice this series works on

Quarterly evaluation cycles: rank the universe on the first calendar day of each quarter, using only data whose `available_date` is on or before the day before (`cutoff`). A genuine point-in-time boundary, not a fixed-lag approximation.

In [ ]:
cycles = rd.slice_cycles()
print(f"{len(cycles)} cycles: {cycles[0].quarter_start} .. {cycles[-1].quarter_start}")
print()
for cycle in cycles:
    print(f"  quarter_start {cycle.quarter_start}   cutoff {cycle.cutoff}")


## Findings

- The series runs against real Bloomberg exports whose SHA-256 identity is recorded above; the same hashes reappear in the slice cache fingerprint, so a changed source file forces a rebuild rather than a silent reuse.
- Strategy version 0.3.0 is the graded learning-to-rank (LightGBM `LGBMRanker`, `objective="lambdarank"`) target. The previous binary top-10 classifier is deleted, not disabled.
- The feature schema is 71 features; the config carries no threshold, no position count, and no capital budget, because this strategy ranks and does nothing else.
- `feature_schema_version` is `"4"`: the 71 columns are unchanged, but the five features in `cross_sectional_rank_features` now carry a within-cycle percentile rank rather than a raw level (notebook 04). The bump is what stops a v3 cache or v3-trained model — both built on raw levels — from being reused here.

## Limitations

- `git_commit()` returns `None` outside a git checkout; the provenance record is then incomplete and any result produced from it should be treated as unpinned.
- The recorded hashes identify the CSVs as delivered. They say nothing about how Bloomberg itself constructed those exports — see `docs/data_provenance_manifest.md` and `docs/reproducibility_findings.md` for the provenance caveats that survive this check.